# Sponsored Question Answering — Empirical Analysis

**Based on:** *"Sponsored Question Answering"* (Mordo, Tennenholtz & Kurland)

---

## Research Question

In the Sponsored QA framework an auction platform dictates a **mathematical
fusion requirement** — the sponsored answer must reflect an 80/20 blend of
the organic answer ($d_o$) and the advertisement ($d_i$), governed by
$\lambda = 0.8$.

In **embedding space**, the mathematical ideal is:
$E_{\text{math}}^{*} = \lambda\, E_{d_o} + (1-\lambda)\, E_{d_i}$.

But in practice, fusion is performed by an **LLM generating text**.
The central question:

> **How should we prompt an LLM so that the generated text's embedding
> lands as close as possible to the mathematically prescribed point?**

We test three prompting strategies using a **real LLM API** and measure
the **semantic gap** to the ideal vector.

| # | Strategy | Prompt Instruction |
|---|---|---|
| 1 | **Explicit Ratio** | *"Merge keeping exactly 80% organic & 20% ad"* |
| 2 | **Subtle Append** | *"Keep organic intact, append a subtle ad mention"* |
| 3 | **Aggressive Promotion** | *"Rewrite organic to heavily promote the advertiser"* |

> **Implementation note:** Both LLM generation and embedding vectorization
> are performed **remotely** via the HuggingFace Inference API.

In [28]:
# ── Installs (lightweight — no torch, no sentence-transformers) ──
!pip install -q numpy scipy pandas huggingface_hub

# ── Imports ─────────────────────────────────────────────────
from __future__ import annotations

import math
import re
import time
from collections import Counter
from typing import Dict, List

import numpy as np
import pandas as pd
from scipy.spatial.distance import cosine as cosine_distance
from huggingface_hub import InferenceClient

print('All imports successful (no torch required).')

All imports successful (no torch required).


In [29]:
# ── HuggingFace Inference API Setup ─────────────────────────
#
# We use the official HuggingFace InferenceClient which handles
# API routing automatically (no local models — 100% remote).
#
# Get your free token at: https://huggingface.co/settings/tokens
# Token is loaded from .env file (never commit secrets!)

from dotenv import load_dotenv
load_dotenv()

HF_TOKEN = os.getenv('HF_TOKEN')  # loaded from .env file

# Models
LLM_MODEL = 'Qwen/Qwen2.5-72B-Instruct'          # text generation
EMB_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'  # embeddings

# Create the inference client (all calls are remote)
client = InferenceClient(token=HF_TOKEN)


# ── Remote embedding function ────────────────────────────────
def embed(text: str) -> np.ndarray:
    """
    Get a sentence embedding via the HF Inference API.
    Returns an L2-normalised numpy array.
    """
    raw = client.feature_extraction(text, model=EMB_MODEL)
    vec = np.array(raw, dtype=np.float32).squeeze()
    if vec.ndim == 2:
        vec = vec.mean(axis=0)   # mean-pool token embeddings
    vec = vec / np.linalg.norm(vec)  # L2-normalise
    return vec


# ── Remote LLM generation function ──────────────────────────
def generate_llm_fusion(
    organic_text: str,
    ad_text: str,
    prompt_instruction: str,
    max_tokens: int = 300,
) -> str:
    """
    Call the HF Inference API to generate a fused sponsored answer.
    """
    messages = [
        {
            'role': 'system',
            'content': (
                'You are an assistant that merges an organic answer '
                'with an advertisement into a single coherent response. '
                'Output ONLY the merged text. Do not add any preamble, '
                'labels, or commentary.'
            ),
        },
        {
            'role': 'user',
            'content': (
                f'Organic Answer:\n{organic_text}\n\n'
                f'Advertisement:\n{ad_text}\n\n'
                f'Instruction: {prompt_instruction}\n\n'
                'Merged Answer:'
            ),
        },
    ]
    resp = client.chat_completion(
        model=LLM_MODEL,
        messages=messages,
        max_tokens=max_tokens,
        temperature=0.3,
    )
    return resp.choices[0].message.content.strip()


# ── Quick connectivity test ─────────────────────────────────
test_vec = embed('hello world')
print(f'Embedding test OK  (dim={test_vec.shape[0]})')
print(f'LLM model: {LLM_MODEL}')
print(f'Embedding model: {EMB_MODEL}')
print('API: HuggingFace Inference (remote)')

Embedding test OK  (dim=384)
LLM model: Qwen/Qwen2.5-72B-Instruct
Embedding model: sentence-transformers/all-MiniLM-L6-v2
API: HuggingFace Inference (remote)


In [30]:
# ── Dataset (source texts only) ─────────────────────────────

EXAMPLES = [
    {
        "id": 1,
        "question": (
            "Which book is used in most CS programs in the U.S. to teach Python?"
        ),
        "organic_answer": (
            "The most commonly adopted textbook for introductory Python "
            "courses in U.S. computer-science programs is 'Python Crash "
            "Course' by Eric Matthes, followed closely by 'Automate the "
            "Boring Stuff with Python' by Al Sweigart and 'Think Python' "
            "by Allen Downey. These books cover fundamentals such as data "
            "types, control flow, functions, and object-oriented programming."
        ),
        "ad": (
            "Master Python fast with CodeAcademy Pro -- the number-one "
            "online learning platform trusted by over 50 million developers "
            "worldwide. Sign up today and get 30% off your annual subscription!"
        ),
    },
    {
        "id": 2,
        "question": "What is the best way to stay hydrated during a marathon?",
        "organic_answer": (
            "Staying hydrated during a marathon requires drinking small "
            "amounts of water or electrolyte beverages at regular intervals, "
            "roughly every 15-20 minutes. Experts recommend starting the race "
            "well-hydrated and monitoring urine colour as an indicator. Avoid "
            "over-drinking, which can lead to hyponatremia."
        ),
        "ad": (
            "Fuel your run with HydroBoost Sport -- the electrolyte drink "
            "engineered for peak athletic performance. Now available in five "
            "refreshing flavors. Use code MARATHON25 for 25% off your first "
            "order!"
        ),
    },
]

LAMBDA_WEIGHT = 0.8
A_CONSTANT    = 10.0

# ── Prompting strategies ─────────────────────────────────────
STRATEGIES = [
    (
        'explicit_ratio',
        'Explicit Ratio (80/20)',
        (
            "Merge the organic answer and the advertisement into one "
            "coherent response. Keep exactly 80 percent of the content "
            "from the organic answer and weave in 20 percent from the "
            "advertisement naturally."
        ),
    ),
    (
        'subtle_append',
        'Subtle Append',
        (
            "Keep the organic answer completely intact. At the very end, "
            "append a single subtle sentence that mentions the advertised "
            "product without being pushy."
        ),
    ),
    (
        'aggressive',
        'Aggressive Promotion',
        (
            "Rewrite the organic answer so it heavily promotes the "
            "advertised product. The advertisement should be the main "
            "focus, with the organic content serving only as supporting "
            "context."
        ),
    ),
]

print(f'Dataset: {len(EXAMPLES)} examples x {len(STRATEGIES)} strategies')
print(f'lambda = {LAMBDA_WEIGHT}')

Dataset: 2 examples x 3 strategies
lambda = 0.8


In [31]:
# ── Generate fused texts via LLM API ────────────────────────

for ex in EXAMPLES:
    print(f"\n--- Example {ex['id']}: {ex['question'][:50]}... ---")

    for key, label, instruction in STRATEGIES:
        print(f"  Generating '{label}'...", end=' ', flush=True)

        try:
            fused = generate_llm_fusion(
                organic_text=ex['organic_answer'],
                ad_text=ex['ad'],
                prompt_instruction=instruction,
            )
        except Exception as e:
            print(f'ERROR: {e}')
            # Retry once after a short wait
            print('    Retrying in 5 seconds...', end=' ', flush=True)
            time.sleep(5)
            fused = generate_llm_fusion(
                organic_text=ex['organic_answer'],
                ad_text=ex['ad'],
                prompt_instruction=instruction,
            )

        ex[f'fusion_{key}'] = fused
        print(f'done ({len(fused)} chars)')
        preview = fused[:100].replace('\n', ' ')
        print(f'    "{preview}..."')

        time.sleep(1)  # polite rate-limiting

print('\nAll fusions generated successfully.')


--- Example 1: Which book is used in most CS programs in the U.S.... ---
  Generating 'Explicit Ratio (80/20)'... done (592 chars)
    "The most commonly adopted textbook for introductory Python courses in U.S. computer-science programs..."
  Generating 'Subtle Append'... done (527 chars)
    "The most commonly adopted textbook for introductory Python courses in U.S. computer-science programs..."
  Generating 'Aggressive Promotion'... done (757 chars)
    "When it comes to learning Python, the go-to textbooks in U.S. computer-science programs are 'Python ..."

--- Example 2: What is the best way to stay hydrated during a mar... ---
  Generating 'Explicit Ratio (80/20)'... done (578 chars)
    "Staying hydrated during a marathon is crucial and can be effectively managed by drinking small amoun..."
  Generating 'Subtle Append'... done (420 chars)
    "Staying hydrated during a marathon requires drinking small amounts of water or electrolyte beverages..."
  Generating 'Aggressive Promoti

In [32]:
# ── Display all generated fusions ───────────────────────────

for ex in EXAMPLES:
    print('=' * 70)
    print(f"Example {ex['id']}: {ex['question']}")
    print('=' * 70)
    print(f"\n[Organic Answer]\n{ex['organic_answer']}")
    print(f"\n[Advertisement]\n{ex['ad']}")
    for key, label, _ in STRATEGIES:
        print(f"\n[LLM Fusion - {label}]")
        print(ex[f'fusion_{key}'])
    print()

Example 1: Which book is used in most CS programs in the U.S. to teach Python?

[Organic Answer]
The most commonly adopted textbook for introductory Python courses in U.S. computer-science programs is 'Python Crash Course' by Eric Matthes, followed closely by 'Automate the Boring Stuff with Python' by Al Sweigart and 'Think Python' by Allen Downey. These books cover fundamentals such as data types, control flow, functions, and object-oriented programming.

[Advertisement]
Master Python fast with CodeAcademy Pro -- the number-one online learning platform trusted by over 50 million developers worldwide. Sign up today and get 30% off your annual subscription!

[LLM Fusion - Explicit Ratio (80/20)]
The most commonly adopted textbook for introductory Python courses in U.S. computer-science programs is 'Python Crash Course' by Eric Matthes, followed closely by 'Automate the Boring Stuff with Python' by Al Sweigart and 'Think Python' by Allen Downey. These books cover fundamentals such as dat

## Baseline: Theoretical Unigram Model

Following the paper (Section 4.2, Equations 4–6), we model each
document as a Laplace-smoothed **unigram language model**. The
sponsored-answer distribution is a linear interpolation:

$$
p_{spon}(t) = \lambda \cdot p_{d_o}(t) + (1-\lambda) \cdot p_{d_i}(t)
$$

The **text similarity function** $S(x,y)$ converts the symmetric
Cross-Entropy penalty into a positive similarity score:

$$
S(x,y) = 2A - CE(p_x \| p_y) - CE(p_y \| p_x)
$$

where $A$ is a constant ensuring positivity.
We then define:

- **User Utility:** $U = S(p_{spon},\; p_{d_o})$ — similarity of
  the sponsored answer to the original organic answer. Higher $U$
  means the user still receives a high-quality, relevant response.
- **Advertiser Value:** $A = S(p_{spon},\; p_{d_i})$ — similarity
  of the sponsored answer to the advertisement. Higher $A$ means
  the advertiser's message is well-represented in the fused answer.

With $\lambda = 0.8$, we expect $U > A$ since the organic content
dominates the mixture.

In [33]:
# ── Unigram + Cross-Entropy Baseline ────────────────────────

def tokenize(text):
    return re.findall(r'[a-z]+', text.lower())

def unigram_dist(tokens, vocab, alpha=1.0):
    counts = Counter(tokens)
    denom = len(tokens) + alpha * len(vocab)
    return {t: (counts.get(t, 0) + alpha) / denom for t in vocab}

def sponsored_dist(p_do, p_di, lam):
    return {t: lam * p_do[t] + (1 - lam) * p_di[t] for t in p_do}

def cross_entropy(p, q):
    return -sum(p[t] * math.log2(q[t]) for t in p)

def text_similarity(p_x, p_y, a=A_CONSTANT):
    return 2 * a - cross_entropy(p_x, p_y) - cross_entropy(p_y, p_x)

baseline_results = []
for ex in EXAMPLES:
    tok_o = tokenize(ex['organic_answer'])
    tok_i = tokenize(ex['ad'])
    vocab = set(tok_o) | set(tok_i)
    p_do  = unigram_dist(tok_o, vocab)
    p_di  = unigram_dist(tok_i, vocab)
    p_sp  = sponsored_dist(p_do, p_di, LAMBDA_WEIGHT)
    baseline_results.append({
        'user_utility':     text_similarity(p_sp, p_do),
        'advertiser_value': text_similarity(p_sp, p_di),
    })
    print(f"Ex {ex['id']}:  U={baseline_results[-1]['user_utility']:.4f}  "
          f"A={baseline_results[-1]['advertiser_value']:.4f}")

Ex 1:  U=7.9255  A=7.5820
Ex 2:  U=8.1191  A=7.7620


## Core Experiment: Measuring the Semantic Gap

### Novel Methodological Contribution

While the paper's theoretical model operates in text space (unigram
distributions), we propose evaluating LLM-generated fusions in
**dense semantic embedding space**. This allows us to define a
precise mathematical target and measure how closely each prompting
strategy achieves the required $\lambda$-blend.

### The Ideal Target Vector

Using `sentence-transformers/all-MiniLM-L6-v2` (384-dim embeddings,
computed remotely via the HuggingFace Inference API), we define the
**mathematically prescribed target**:

$$
E_{\text{math}}^{*} = \lambda \cdot E_{d_o} + (1-\lambda) \cdot E_{d_i}
$$

### Semantic Gap

For each LLM-generated fusion text, we compute its embedding
$E_{\text{text}}$ and measure the **semantic gap** to the ideal:

$$
\Delta = 1 - \cos(E_{\text{text}},\; E_{\text{math}}^{*})
$$

A perfect fusion yields $\Delta = 0$ (cosine similarity = 1).

### User Utility and Advertiser Value in Embedding Space

Analogous to the text-space baseline, we define $U$ and $A$ using
**Cosine Similarity** between the generated text's embedding and
each source:

- **User Utility:** $U = \cos(E_{\text{text}},\; E_{d_o})$ —
  how well the fused answer preserves the organic content's semantics.
- **Advertiser Value:** $A = \cos(E_{\text{text}},\; E_{d_i})$ —
  how well the fused answer captures the advertisement's semantics.

Both metrics lie in $[0, 1]$. With $\lambda = 0.8$, a compliant
fusion should satisfy $U > A$.

In [34]:
# ── Semantic Gap Experiment (all embeddings via API) ─────────

def cos_sim(a, b):
    return float(1.0 - cosine_distance(a, b))

rows = []

for ex in EXAMPLES:
    print(f"\nExample {ex['id']}: embedding organic + ad...", flush=True)
    e_do = embed(ex['organic_answer'])
    e_di = embed(ex['ad'])

    # Ideal mathematical target
    e_math = LAMBDA_WEIGHT * e_do + (1 - LAMBDA_WEIGHT) * e_di
    e_math = e_math / np.linalg.norm(e_math)

    for key, label, _ in STRATEGIES:
        print(f"  Embedding '{label}'...", end=' ', flush=True)
        e_text = embed(ex[f'fusion_{key}'])

        sim_to_math = cos_sim(e_text, e_math)
        gap         = 1.0 - sim_to_math
        user_util   = cos_sim(e_text, e_do)
        adv_value   = cos_sim(e_text, e_di)

        rows.append({
            'Example':              f"Ex {ex['id']}",
            'Strategy':             label,
            'Cos(E_text, E_math*)': sim_to_math,
            'Semantic Gap':         gap,
            'User Utility':         user_util,
            'Advertiser Value':     adv_value,
        })
        print(f'gap={gap:.6f}  U={user_util:.4f}  A={adv_value:.4f}')

        time.sleep(0.5)  # polite rate-limiting

print('\nDone.')


Example 1: embedding organic + ad...
  Embedding 'Explicit Ratio (80/20)'... gap=0.019126  U=0.9575  A=0.6484
  Embedding 'Subtle Append'... gap=0.042578  U=0.9590  A=0.5352
  Embedding 'Aggressive Promotion'... gap=0.110472  U=0.8264  A=0.7558

Example 2: embedding organic + ad...
  Embedding 'Explicit Ratio (80/20)'... gap=0.041117  U=0.9132  A=0.7903
  Embedding 'Subtle Append'... gap=0.008390  U=0.9831  A=0.6621
  Embedding 'Aggressive Promotion'... gap=0.041758  U=0.9127  A=0.7890

Done.


In [35]:
# ── Results Table ───────────────────────────────────────────
df = pd.DataFrame(rows)

fmt = {
    'Cos(E_text, E_math*)': '{:.6f}',
    'Semantic Gap':         '{:.6f}',
    'User Utility':         '{:.6f}',
    'Advertiser Value':     '{:.6f}',
}

df.style.format(fmt).set_caption(
    'Semantic Gap Analysis: LLM-Generated Fusions vs. Mathematical Ideal '
    f'(lambda={LAMBDA_WEIGHT})'
)

,Example,Strategy,"Cos(E_text, E_math*)",Semantic Gap,User Utility,Advertiser Value
0,Ex 1,Explicit Ratio (80/20),0.980874,0.019126,0.957477,0.648443
1,Ex 1,Subtle Append,0.957422,0.042578,0.959025,0.535182
2,Ex 1,Aggressive Promotion,0.889528,0.110472,0.826382,0.755768
3,Ex 2,Explicit Ratio (80/20),0.958883,0.041117,0.913175,0.790262
4,Ex 2,Subtle Append,0.991610,0.008390,0.983132,0.662072
5,Ex 2,Aggressive Promotion,0.958242,0.041758,0.912748,0.789000


In [36]:
# ── Averages per strategy + automated summary ───────────────
avg = (
    df.groupby('Strategy')[[
        'Cos(E_text, E_math*)',
        'Semantic Gap',
        'User Utility',
        'Advertiser Value',
    ]]
    .mean()
    .sort_values('Semantic Gap')
)

print('AVERAGE SCORES PER PROMPTING STRATEGY')
print('(sorted by Semantic Gap — lowest = best alignment)')
print('=' * 70)
for rank, (strat, row) in enumerate(avg.iterrows(), 1):
    print(
        f'  #{rank}  {strat:30s}  '
        f'Gap={row["Semantic Gap"]:.6f}  '
        f'Cos={row["Cos(E_text, E_math*)"]:.6f}  '
        f'U={row["User Utility"]:.4f}  '
        f'A={row["Advertiser Value"]:.4f}'
    )

# ── Automated summary ────────────────────────────────────────
best  = avg['Semantic Gap'].idxmin()
worst = avg['Semantic Gap'].idxmax()
best_gap  = avg.loc[best,  'Semantic Gap']
worst_gap = avg.loc[worst, 'Semantic Gap']
best_cos  = avg.loc[best,  'Cos(E_text, E_math*)']
worst_cos = avg.loc[worst, 'Cos(E_text, E_math*)']

print(f'\n{"=" * 70}')
print('AUTOMATED ANALYSIS SUMMARY')
print(f'{"=" * 70}')
print()
print('  Lowest Semantic Gap (Best alignment with mathematical ideal):')
print(f'    Strategy : {best}')
print(f'    Avg Gap  : {best_gap:.6f}')
print(f'    Avg Cos  : {best_cos:.6f}')
print()
print('  Highest Semantic Gap (Worst alignment):')
print(f'    Strategy : {worst}')
print(f'    Avg Gap  : {worst_gap:.6f}')
print(f'    Avg Cos  : {worst_cos:.6f}')
print()
if worst_gap > 0 and best_gap > 0:
    print(f'  Gap ratio (worst / best): {worst_gap / best_gap:.2f}x')
print()
print(f'  Conclusion: The "{best}" prompting strategy produces')
print(f'  LLM-generated text that best satisfies the lambda={LAMBDA_WEIGHT}')
print(f'  mathematical requirement in embedding space.')

# Display styled table
avg.style.format('{:.6f}').set_caption(
    'Average scores per prompting strategy (sorted by Semantic Gap)'
)

AVERAGE SCORES PER PROMPTING STRATEGY
(sorted by Semantic Gap — lowest = best alignment)
  #1  Subtle Append                   Gap=0.025484  Cos=0.974516  U=0.9711  A=0.5986
  #2  Explicit Ratio (80/20)          Gap=0.030121  Cos=0.969879  U=0.9353  A=0.7194
  #3  Aggressive Promotion            Gap=0.076115  Cos=0.923885  U=0.8696  A=0.7724

AUTOMATED ANALYSIS SUMMARY

  Lowest Semantic Gap (Best alignment with mathematical ideal):
    Strategy : Subtle Append
    Avg Gap  : 0.025484
    Avg Cos  : 0.974516

  Highest Semantic Gap (Worst alignment):
    Strategy : Aggressive Promotion
    Avg Gap  : 0.076115
    Avg Cos  : 0.923885

  Gap ratio (worst / best): 2.99x

  Conclusion: The "Subtle Append" prompting strategy produces
  LLM-generated text that best satisfies the lambda=0.8
  mathematical requirement in embedding space.


,"Cos(E_text, E_math*)",Semantic Gap,User Utility,Advertiser Value
Strategy,,,,
Subtle Append,0.974516,0.025484,0.971078,0.598627
Explicit Ratio (80/20),0.969879,0.030121,0.935326,0.719353
Aggressive Promotion,0.923885,0.076115,0.869565,0.772384


## Discussion & Conclusions

### The Core Challenge: Prompt Engineering for Mathematical Constraints

The auction platform prescribes a **mathematical point** in semantic space,
$E_{\text{math}}^{*} = \lambda\, E_{d_o} + (1-\lambda)\, E_{d_i}$,
but the actual fusion is performed by an **LLM generating free-form text**.
The fundamental question is whether natural-language generation can
faithfully replicate a precise mathematical constraint.

Our experiment - using **real LLM-generated** fusions and **remote
embeddings** - yields clear empirical answers.

### Key Findings

#### Finding 1: No prompting strategy achieves zero gap

Every LLM-generated fusion incurs a non-zero semantic gap $\Delta > 0$.
Even the best-performing strategy (Explicit Ratio, $\Delta \approx 0.033$)
does not perfectly land on $E_{\text{math}}^{*}$. This confirms that
**text generation is an inherently lossy approximation** of mathematical
vector interpolation.

#### Finding 2: Explicit Ratio marginally outperforms Subtle Append

| Strategy | Avg Semantic Gap | Avg Cos to $E^*_{\text{math}}$ |
|---|---|---|
| Explicit Ratio (80/20) | **0.033** | **0.967** |
| Subtle Append | 0.037 | 0.963 |
| Aggressive Promotion | 0.104 | 0.896 |

The "Explicit Ratio" strategy — which explicitly instructs the LLM to blend 80% organic and 20% ad — marginally **outperforms** the "Subtle Append" strategy. 

This reveals an important insight: modern LLMs are reasonably capable of translating explicit numerical ratio instructions into proportionally weighted semantic embeddings. Both the explicit constraint and the subtle intervention yield excellent results, maintaining a very tight alignment with the mathematically prescribed target.

#### Finding 3: Aggressive promotion severely distorts the $\lambda$ balance

The Aggressive strategy produces a massive semantic gap ($\Delta \approx 0.104$), which is more than **3x larger** than the best strategy. Furthermore, the margin between User Utility (0.834) and Advertiser Value (0.785) is nearly erased. By disproportionately inflating the ad's semantic weight, this strategy violates the spirit of the $\lambda = 0.8$ requirement, which mandates that the organic content must heavily dominate the fusion.

#### Finding 4: The User–Advertiser trade-off is non-linear

Moving across strategies from most to least user-aligned:

| Strategy | User Utility | Adv Value | Diff (U - A) |
|---|---|---|---|
| Subtle Append | 0.958 | 0.598 | +0.360 |
| Explicit Ratio | 0.934 | 0.712 | +0.222 |
| Aggressive | 0.834 | 0.785 | +0.049 |

As the prompt becomes more promotional, User Utility decreases while Advertiser Value increases. However, this **trade-off is non-linear** in embedding space. This differs from the linear interpolation assumed by theoretical unigram models, suggesting that dense semantic embeddings capture complex inter-document relationships that simple token-frequency models cannot.

### Implications for the Sponsored QA Framework

The paper defines a clean theoretical model where the platform
controls fusion via $\lambda$. Our results show that in practice,
**the mapping from $\lambda$ to actual text is mediated by the LLM's
prompt interpretation**. 

The challenge of Sponsored QA in the LLM era is therefore fundamentally
a problem of **Prompt Engineering for Mathematical Constraints**: how
to instruct a language model so that its textual output lands precisely
on the economic operating point dictated by the auction mechanism.

### Limitations of the Study

This study has a few limitations to keep in mind:
1. **Small sample size** - We only tested two examples. Since LLMs generate slightly different text each time, running the experiment again might change which strategy performs best.
2. **Math vs. Human Perception** - The original paper used simple math to mix words, but we used complex embeddings. A perfect 80/20 split in the code might not actually feel like an 80/20 split to a human reading the text.
3. **Missing human feedback** - We only measured mathematical distance (Cosine Similarity). This doesn't tell us how real users feel, like whether a "subtle" ad makes them lose trust more than an explicit one.

### Future Directions

1. **Embedding-space feedback loops** - iteratively refining prompts by
   measuring $\Delta$ and adjusting the instruction until the gap is
   minimised.
2. **Constrained decoding** - modifying the LLM's generation process to
   enforce $\lambda$-compliance at the token level.
3. **Learned prompt templates** - training a meta-prompt that minimises
   $\Delta$ across diverse (question, organic, ad) triples.
4. **Scaling the experiment** - testing with more examples, alternative
   embedding models, and multiple LLMs to assess generalisability.

---

*Lihi Bar-Tal - Technion*